In [1]:
import pandas as pd
import json
import random
from collections import Counter

# ===============================
# 경로
# ===============================
tsv_path = r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\CLIP_\filtered_variants_cleaned_final.tsv"
seq_json_path = r"C:\Users\Kunny\research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"
out_path = r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\CLIP_\filtered_variants_augmented.tsv"

# ===============================
# 로드
# ===============================
df = pd.read_csv(tsv_path, sep="\t")
df = df.drop(columns=["Label"], errors="ignore")

with open(seq_json_path, "r") as f:
    id_to_seq = json.load(f)

AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

# ===============================
# UniProtID별 offset 계산
# ===============================
offset_map = {}

for uid, g in df.groupby("UniProtID"):
    offsets = (g["MutPos(pdb)"] - g["MutPos"]).tolist()
    offset_map[uid] = Counter(offsets).most_common(1)[0][0]

# ===============================
# 기존 변이 set (중복 방지)
# ===============================
existing_keys = set(
    zip(df["UniProtID"], df["MutPos"], df["WT"], df["Mut"])
)

# ===============================
# 증강
# ===============================
augmented_rows = []
target_total = len(df) * 4

while len(df) + len(augmented_rows) < target_total:
    row = df.sample(1).iloc[0]
    uid = row["UniProtID"]

    if uid not in id_to_seq:
        continue

    seq = id_to_seq[uid]
    seq_len = len(seq)

    mut_pos = random.randint(1, seq_len)
    wt = seq[mut_pos - 1]
    mut = random.choice([aa for aa in AA_LIST if aa != wt])

    key = (uid, mut_pos, wt, mut)
    if key in existing_keys:
        continue  # 중복이면 다시 뽑기

    pdb_pos = mut_pos + offset_map.get(uid, 0)

    augmented_rows.append({
        "UniProtID": uid,
        "MutPos": mut_pos,
        "WT": wt,
        "Mut": mut,
        "StructureFile": row["StructureFile"],
        "MutPos(pdb)": pdb_pos
    })

    existing_keys.add(key)

# ===============================
# 저장
# ===============================
aug_df = pd.concat([df, pd.DataFrame(augmented_rows)], ignore_index=True)
aug_df.to_csv(out_path, sep="\t", index=False)

print(f"Done: {len(df)} → {len(aug_df)} (duplicates removed, offset preserved)")

KeyboardInterrupt: 